In [1]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_excel('Base_Segmentacion.xlsx')

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72780 entries, 0 to 72779
Columns: 153 entries, Identificacion_anonimizada to Indicador_empleado_Banco
dtypes: float64(45), int64(65), object(43)
memory usage: 85.0+ MB


In [7]:
df.shape

(72780, 153)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')


## 1. Exploración descriptiva

Antes de proponer un modelo de segmentación conviene caracterizar la base: su dimensión, los tipos de datos que contiene y el comportamiento de las variables demográficas y transaccionales más relevantes. Esta exploración permite decidir qué variables aportan valor para agrupar a los usuarios y cuáles conviene excluir por ser identificadores, redundantes o de cardinalidad demasiado alta.


In [ ]:
# Tipos de datos, valores nulos y registros duplicados
print(df.dtypes.value_counts().to_string())
print()
print('Valores nulos por columna:', int(df.isnull().sum().sum()))
print('Registros duplicados:', int(df.duplicated().sum()))
print('Filas, columnas:', df.shape)


In [ ]:
# Distribución de variables demográficas clave
print('Género:\n', df['Genero'].value_counts(), sep='')
print('\nEstrato:\n', df['Estrato'].value_counts().sort_index(), sep='')
print('\nRegión:\n', df['Region'].value_counts(), sep='')


In [ ]:
# Distribución de edad y de saldo en la aplicación
print('Edad:\n', df['Edad_Actual'].describe().round(2), sep='')
print('\nSaldo a enero:\n', df['Saldo_Aplicacion_Enero'].describe().round(2), sep='')
print('\nActividad monetaria:\n', df['Indicador_Transaccion_monetaria'].value_counts(normalize=True).round(4), sep='')


La base tiene 72.780 clientes y 153 variables, con una mezcla de 65 columnas enteras, 45 de coma flotante y 43 de tipo objeto, sin valores nulos ni registros duplicados. La muestra está relativamente equilibrada por género (56,4 % femenino y 43,6 % masculino) y concentrada en los estratos bajos y medios: los estratos 1, 0 y 2 reúnen cerca de 64.000 clientes. La edad media es de 37,5 años con una mediana de 34,7. El saldo a enero muestra una fuerte asimetría positiva: la mediana es cero y la media de 65.947 pesos, lo que indica que la mayor parte de los clientes mantiene saldos nulos o muy bajos y una minoría concentra saldos elevados. En la misma línea, solo el 13,5 % de los usuarios ha realizado transacciones monetarias por la aplicación, un primer indicio de que la actividad transaccional es el eje que más distingue a los clientes entre sí.


## 2. Preparación de los datos

Para construir la matriz de segmentación se descartan las variables que no aportan estructura: el identificador anonimizado, la ciudad (688 valores únicos, redundante con la región), el año de nacimiento (redundante con la edad), el tipo de vivienda (categórica de diez niveles que no aporta al eje transaccional), la marca de microsegmento preexistente (para no condicionar el resultado) y las columnas de rangos, que son recategorizaciones de las variables numéricas de saldo. Las variables de objeto de tipo Sí/No se convierten a indicadores binarios y las categóricas restantes se codifican numéricamente. Finalmente se estandarizan las variables para que las de mayor escala no dominen la distancia euclidiana que usa K-Means.


In [ ]:
# Selección y codificación de variables para la segmentación
excluir = ['Identificacion_anonimizada', 'Ciudad', 'Año_nacimiento',
           'MarcaClienteMicrosegmento', 'TipoVivienda']
excluir += [c for c in df.columns if c.startswith('Rangos_')]

obj = [c for c in df.select_dtypes('object').columns if c not in excluir]
codificadas = []
for c in obj:
    if df[c].nunique() == 2:
        valores = sorted(df[c].dropna().unique())
        mapeo = {v: (1 if str(v).strip().lower() == 'si' else 0) for v in valores}
        codificadas.append(df[c].map(mapeo).astype(float).rename(c))
    else:
        codificadas.append(pd.Series(pd.factorize(df[c])[0], index=df.index, name=c).astype(float))

numericas = [c for c in df.select_dtypes('number').columns if c not in excluir]
X = pd.concat([df[numericas]] + codificadas, axis=1)

X_std = StandardScaler().fit_transform(X.fillna(0))
print('Dimensiones de la matriz de segmentación:', X_std.shape)


## 3. Segmentación con K-Means

Se evalúa el número de conglomerados mediante el método del codo (inercia) y el coeficiente de silueta, que mide qué tan compactos y separados están los grupos. La inercia se calcula sobre toda la muestra, mientras que la silueta se estima sobre una submuestra de 10.000 clientes para reducir el costo computacional.


In [ ]:
# Método del codo: inercia para k entre 2 y 8
inercia = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_std)
    inercia.append(km.inertia_)
print('Inercia por k:')
for k, val in zip(range(2, 9), inercia):
    print(f'  k={k}: {val:.0f}')

plt.figure(figsize=(8, 5))
plt.plot(range(2, 9), inercia, marker='o')
plt.xlabel('Número de conglomerados (k)')
plt.ylabel('Inercia')
plt.title('Método del codo')
plt.grid(True)
plt.show()


In [ ]:
# Coeficiente de silueta sobre una submuestra de 10.000 clientes
rng = np.random.RandomState(42)
indices = rng.choice(len(X_std), size=10000, replace=False)
X_sub = X_std[indices]

siluetas = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    etiquetas = km.fit_predict(X_sub)
    sil = silhouette_score(X_sub, etiquetas, sample_size=3000, random_state=42)
    siluetas.append(sil)
    print(f'  k={k}: silueta={sil:.4f}')

plt.figure(figsize=(8, 5))
plt.plot(range(2, 9), siluetas, marker='o')
plt.xlabel('Número de conglomerados (k)')
plt.ylabel('Coeficiente de silueta')
plt.title('Silueta por número de conglomerados')
plt.grid(True)
plt.show()


La silueta alcanza su valor máximo con dos conglomerados (0,94), muy por encima de cualquier otra configuración, mientras que la inercia desciende de forma continua sin un codo pronunciado. Este resultado revela que la estructura dominante de los datos es una separación limpia entre dos perfiles: clientes con actividad transaccional y saldos relevantes frente a clientes inactivos o con saldo marginal. Al aumentar k los conglomerados adicionales resultan muy pequeños y desbalanceados, lo que indica que la base no contiene muchas tipologías de comportamiento bien diferenciadas más allá de ese eje principal.


In [ ]:
# Ajuste con k = 2 y perfil de cada conglomerado
km2 = KMeans(n_clusters=2, random_state=42, n_init=10)
df['segmento'] = km2.fit_predict(X_std)

perfil_cols = ['Edad_Actual', 'Saldo_Aplicacion_Enero', 'Saldo_Promedio_12M',
               'Cantidad_productos_banco', 'Cantidad_giros', 'Tiempo_vida_aplicacion',
               'TiempoVidaBanco', 'Cantidad_modulos_6m']
df['tx_activo'] = (df['Indicador_Transaccion_monetaria'] == 'Si').astype(int)
perfil = df.groupby('segmento')[perfil_cols + ['tx_activo']].mean().round(2)
perfil['n'] = df['segmento'].value_counts().sort_index()
print(perfil)


## 4. Refinamiento dentro del segmento activo

Como el primer nivel separa esencialmente usuarios activos de inactivos, se profundiza el análisis sobre el grupo que sí realiza transacciones monetarias para detectar tipologías más finas. Se aplica K-Means con tres conglomerados únicamente sobre esos clientes activos.


In [ ]:
# Segmentación de los usuarios con actividad monetaria
activos = df['Indicador_Transaccion_monetaria'] == 'Si'
X_act = X_std[activos.values]

km_act = KMeans(n_clusters=3, random_state=42, n_init=10)
df.loc[activos, 'subsegmento'] = km_act.fit_predict(X_act)

sub_perfil = df[activos].groupby('subsegmento')[perfil_cols].mean().round(2)
sub_perfil['n'] = df[activos]['subsegmento'].value_counts().sort_index()
print(sub_perfil)


## 5. Conclusiones y justificación del modelo

La segmentación propuesta se estructura en dos niveles. El primer nivel, respaldado por el coeficiente de silueta de 0,94, distingue a los clientes activos (6.037, con un saldo a enero cercano a 300.000 pesos y transaccionalidad monetaria plena) de los inactivos (66.743, con saldos casi nulos y sin operaciones monetarias). Este es el corte con mayor valor de negocio, porque permite concentrar las estrategias de activación y de retención en cada grupo. El segundo nivel refina los 9.806 usuarios que sí registran transacciones monetarias y los separa en tres subgrupos: uno mayoritario (9.778) de actividad moderada con un saldo a enero en torno a 190.000 pesos, un grupo reducido (26) de saldo alto y mayor antigüedad con saldos cercanos a 686.000 pesos, y una cola extrema de dos usuarios cuyo saldo supera los 3,5 millones de pesos.

La principal limitación del modelo es que, al exigir más conglomerados, estos se vuelven muy pequeños y desbalanceados, lo que sugiere que la variabilidad de comportamiento se concentra sobre todo en la intensidad de uso y el saldo, más que en tipologías cualitativamente distintas. Convendría, en trabajos futuros, incorporar variables de temporalidad de las transacciones a distintos cortes y comparar con métodos que toleren conglomerados de tamaño heterogéneo, como los modelos basados en densidad o en mezclas gaussianas.
